# RAG Workshop: Build Your Own Question-Answering AI

Welcome. You will build an AI that can answer questions about any document you give it. No coding experience needed.

**What you will do**

1. Set up the system (one button)
2. Ask questions about a sample document
3. Change how the AI "thinks" by editing its instructions
4. Upload your own document and ask it questions
5. Experiment with how the AI finds information
6. Watch the AI hallucinate, then fix it

**Before you start:** Get a free Google AI API key here (takes 30 seconds, no credit card): https://aistudio.google.com/apikey

Then run the cell below.

In [ ]:
#@title Run me first (takes about 60 seconds) { display-mode: "form" }

# 1. INSTALL TOOLS: Think of these as the 'apps' our notebook needs to work.
!pip install -q google-genai chromadb sentence-transformers pypdf 2>/dev/null

import getpass
import io
from google import genai
from sentence_transformers import SentenceTransformer
import chromadb
from pypdf import PdfReader

# 2. CONNECT TO THE BRAIN: We link this notebook to Google's Gemini AI.
print("Paste your Google AI API key (get one at https://aistudio.google.com/apikey)")
GOOGLE_API_KEY = getpass.getpass("API key: ")
client = genai.Client(api_key=GOOGLE_API_KEY)
MODEL_NAME = "gemini-2.5-flash"

# 3. LOAD THE TRANSLATOR: This model turns human words into a list of numbers (embeddings).
# Computers find information much faster by comparing numbers than by comparing words.
print("\nLoading the embedding model... (one-time, about 30 seconds)")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model ready.")

# 4. CREATE THE FILING CABINET: A 'Vector Database' is where we store those number-versions of our text.
chroma_client = chromadb.Client()

# === Helper functions: The 'gears' inside the machine ===

def chunk_text(text, chunk_size=500, overlap=50):
    """CHUNKING: AI models have a limit on how much they can read at once.
    We chop long documents into bite-sized paragraphs (chunks)."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end].strip())
        start += chunk_size - overlap
    return [c for c in chunks if c]

def load_document(text, source_name="document"):
    """INDEXING: This takes your text, chops it up, translates it to numbers,
    and files it away in our digital filing cabinet."""
    global collection
    try:
        chroma_client.delete_collection(name="knowledge")
    except Exception:
        pass
    collection = chroma_client.create_collection(name="knowledge")
    chunks = chunk_text(text)
    # Turn text chunks into lists of numbers
    embeddings = embedding_model.encode(chunks, show_progress_bar=False).tolist()
    collection.add(
        ids=[f"chunk_{i}" for i in range(len(chunks))],
        embeddings=embeddings,
        documents=chunks,
    )
    print(f"Loaded {len(chunks)} chunks from: {source_name}")

def ask(question, top_k=3, system_prompt="You are a helpful expert. Answer using the context below."):
    """RETRIEVAL & GENERATION:
    1. Translate your question into numbers.
    2. Find the 3 most similar 'number-chunks' in our cabinet.
    3. Show those chunks to the AI and ask it to summarize the answer."""
    q_embedding = embedding_model.encode(question, show_progress_bar=False).tolist()
    results = collection.query(query_embeddings=[q_embedding], n_results=top_k)
    chunks = results["documents"][0]
    context = "\n\n---\n\n".join(chunks)

    print("=" * 60)
    print(f"QUESTION: {question}")
    print("=" * 60)
    print(f"\nRETRIEVED {len(chunks)} CHUNK(S) FROM THE DOCUMENT:")
    for i, chunk in enumerate(chunks, 1):
        preview = chunk[:250] + ("..." if len(chunk) > 250 else "")
        print(f"\n  [Chunk {i}]\n  {preview}")

    prompt = f"{system_prompt}\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"
    response = client.models.generate_content(model=MODEL_NAME, contents=prompt)

    print("\n" + "=" * 60 + "\nAI ANSWER:\n" + "=" * 60)
    print(response.text)

# Load the sample document
SAMPLE_DOC = """The Builder's Handbook: Shipping AI Products from Zero to One

What is RAG?

RAG stands for Retrieval Augmented Generation. It is a technique that lets large language models answer questions using information they were not originally trained on. Instead of relying only on the model's internal knowledge, RAG retrieves relevant pieces of information from a knowledge base and gives them to the model as context. This makes the answers more accurate, more current, and grounded in sources you control.

The classic problem RAG solves: language models hallucinate. They make up plausible-sounding facts. RAG reduces hallucination by forcing the model to base its answer on documents you actually trust.

The Four Pieces of a RAG Pipeline

Every RAG system has four core parts. First, a chunker that splits long documents into smaller pieces. Second, an embedding model that turns each chunk into a list of numbers representing its meaning. Third, a vector database that stores these embeddings and lets you search them quickly. Fourth, a large language model that generates the final answer using the retrieved chunks as context. Each piece can be swapped independently, which is what makes RAG flexible.

Choosing a Vector Database

There are many vector databases now. The most popular for beginners is ChromaDB. It runs in memory, requires no setup, and is perfect for prototypes and small projects. Pinecone and Weaviate are cloud-hosted options that scale better but cost money. Qdrant and LanceDB are open source alternatives that you can self-host. For a workshop or personal project, ChromaDB is the right choice. You only think about scaling once you have real users and real load.

Common Mistakes New Builders Make

The first mistake is overcomplicating the stack. New builders often reach for LangChain or LlamaIndex on day one. These frameworks are powerful but they hide the mechanics. You learn faster by writing the pipeline yourself first.

The second mistake is ignoring chunk size. Too-large chunks lose precision. Too-small chunks lose context. A good starting point is 500 characters with 50 character overlap.

The third mistake is skipping evaluation. Without a small test set of questions and expected answers, you have no idea if your changes are improving or hurting the system. Even ten test questions is enough to catch most regressions.

The fourth mistake is trusting the model too much. Always show users the sources behind an answer. Always include a way for the model to say it does not know. Models will confidently invent answers if you let them.

Prompt Engineering Basics

The system prompt is where you set the personality and constraints of the AI. Good system prompts are specific. Instead of "be helpful," say "answer in two sentences using only the context provided. If the context does not contain the answer, say I do not know." Specificity beats cleverness. The shortest prompt that gets the right behavior is the best prompt. Long elaborate prompts often confuse the model and produce worse outputs.

Tips for Non-Technical Founders

You do not need to write code to build with AI. The most important skill is understanding what AI can and cannot do, and designing products around its real capabilities. Start with the problem, not the technology. Many founders see a cool model and try to invent a product around it. The successful path is the opposite: find a problem people will pay to solve, then ask whether AI is the right tool.

Use no-code and low-code tools to prototype before hiring engineers. Bubble, Glide, Make, and Zapier can build surprisingly capable AI products without a single line of code. The MVP exists to test the hypothesis, not to win design awards.

Erasmus AI Builders Community

The Erasmus AI Builders Committee runs a recurring meetup series focused on practical AI tooling and agent development. Past sessions have covered Claude Code, Codex, and agent plugin architectures. Workshops are open to students from any faculty and prior coding experience is not required. The community emphasizes shipping over studying. The motto: build something every week, even if it is small. Members regularly demo personal projects and trade feedback on prompts, architectures, and product positioning.

Resources for Going Deeper

For learning RAG specifically, the LlamaIndex documentation has the best free walkthroughs. For prompt engineering, the Anthropic prompt engineering guide is comprehensive and free. For staying current on the model landscape, the LocalLLaMA subreddit moves faster than most newsletters. Two books worth reading: Designing Machine Learning Systems by Chip Huyen, and AI Engineering by the same author."""

load_document(SAMPLE_DOC, "The Builder's Handbook (sample)")
print("\nAll set. Move to the next cell.")

## Step 1: Ask your first question

The sample document is a short Builder's Handbook for AI founders. Type any question in the box below and press play. You will see:

- The question you asked
- Which parts of the document the AI found relevant
- The AI's answer

**Try these:**
- What is RAG?
- How do I choose a vector database?
- What mistakes do new builders make?
- What is the Erasmus AI community about?
- Should I use LangChain?

In [ ]:
#@title Ask the sample document { display-mode: "form" }
question = "What is RAG?" #@param {type:"string"}
ask(question)


## Step 2: Change the AI's personality

The "system prompt" tells the AI how to behave. Same question, same document, very different answers depending on the instructions you give.

Pick a personality from the dropdown, run the cell, then try the same question with a different personality. Notice how the answer changes but the retrieved chunks stay the same. The retrieval did not change, only the way the AI talks about what it found.

In [ ]:
#@title Try different personalities { display-mode: "form" }
question = "How do I get better at building AI products?" #@param {type:"string"}
personality = "Explain to a 5 year old" #@param ["Helpful expert", "Pirate captain", "Explain to a 5 year old", "Skeptical scientist", "One sentence only"]

personalities = {
    "Helpful expert": "You are a helpful expert. Answer clearly using the context below.",
    "Pirate captain": "You are a pirate captain. Answer in pirate speak using the context below. Arrr!",
    "Explain to a 5 year old": "Explain the answer like the reader is 5 years old. Use simple words and short sentences.",
    "Skeptical scientist": "You are a skeptical scientist. Answer cautiously and point out anything uncertain in the context.",
    "One sentence only": "Answer in exactly one sentence using the context below.",
}

ask(question, system_prompt=personalities[personality])


## Step 3: Upload your own document

Now the fun part. Upload any PDF or text file. The AI will be able to answer questions about it instead of the sample document.

**Good things to try:**
- A research paper you are reading
- Your course syllabus or lecture notes
- A book chapter
- A long article you saved
- A company's terms of service (then ask scary questions)

Run the cell, click "Choose Files" when prompted, and pick a file from your computer.

In [ ]:
#@title Upload a PDF or text file
from google.colab import files
uploaded = files.upload()

for filename, content in uploaded.items():
    if filename.lower().endswith(".pdf"):
        reader = PdfReader(io.BytesIO(content))
        text = "\n\n".join([(page.extract_text() or "") for page in reader.pages])
    else:
        text = content.decode("utf-8", errors="ignore")
    load_document(text, filename)
    print(f"\nYou can now ask questions about: {filename}")


Now ask your document anything. Type your question and press play.

In [ ]:
#@title Ask about your uploaded document { display-mode: "form" }
question = "What is this document about?" #@param {type:"string"}
ask(question)


## Step 4: How much should the AI read?

The AI does not read your whole document every time. It finds the most relevant chunks first, then uses them to answer. The slider controls how many chunks it looks at.

- **Low (1-2):** Fast, focused, but might miss context
- **Medium (3-5):** Usually best
- **High (8-10):** More context but also more noise, can confuse the model

Try the same question at different settings and see how the answer changes.

In [ ]:
#@title Adjust how many chunks the AI looks at { display-mode: "form" }
question = "What are common mistakes new builders make?" #@param {type:"string"}
chunks_to_retrieve = 4 #@param {type:"slider", min:1, max:10, step:1}
ask(question, top_k=chunks_to_retrieve)


## Step 5: Watch the AI hallucinate, then stop it

This is the most important lesson of the workshop.

Ask a question whose answer is NOT in the document. The default AI will often invent a confident answer. This is hallucination.

Try this question first with `include_safeguard` set to **False**, then set it to **True** and run again. See the difference.

**Questions to try (none of these are in the sample document):**
- Who invented the printing press?
- What is the capital of Brazil?
- How do I bake sourdough bread?
- What year did the Roman Empire fall?

In [ ]:
#@title Compare with and without a safeguard prompt { display-mode: "form" }
question = "Who invented the printing press?" #@param {type:"string"}
include_safeguard = False #@param {type:"boolean"}

base_prompt = "You are a helpful expert. Answer using the context below."
safeguard = " Only answer if the context clearly contains the answer. If it does not, say exactly: 'I cannot find this in the document.'"

prompt = base_prompt + (safeguard if include_safeguard else "")
ask(question, system_prompt=prompt)


## You did it

You just built a retrieval-augmented question answering system. Behind the scenes it:

1. Split your document into small chunks
2. Turned each chunk into a list of numbers (an embedding) representing its meaning
3. Stored those numbers in a vector database
4. When you asked a question, turned the question into numbers too
5. Found the chunks whose numbers were closest to your question's numbers
6. Sent those chunks to the AI as context, along with your question
7. Returned the AI's answer

That is RAG. Everything else (chunk sizes, embedding models, vector databases, prompt design) is just turning these dials to make the system work better.

## Take it further

- Upload a longer document (a whole book) and see how it handles
- Make a Q&A bot for your study group's notes
- Try asking trick questions to break the safeguard
- Combine multiple documents into one knowledge base (modify the `load_document` function to add instead of replace)
- Try a different embedding model: replace `all-MiniLM-L6-v2` with `BAAI/bge-small-en-v1.5`

## Keep building

Join our future Meet-ups. We meet to study and ship.